# Causal Masked Self-Attention with NumPy
Pure NumPy implementation of stable softmax, a causal additive mask, and scaled dot-product attention.

In [1]:
import numpy as np

Q = np.array([[1., 0., 0.],
              [0., 1., 0.],
              [1., 1., 0.],
              [0., 1., 1.]])

K = np.array([[1., 0., 1.],
              [0., 1., 0.],
              [1., 1., 0.],
              [0., 0., 1.]])

V = np.array([[1., 2.],
              [0., 1.],
              [2., 0.],
              [1., 1.]])

In [2]:
def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z_max = np.max(z, axis=axis, keepdims=True)
    exp_z = np.exp(z - z_max)
    return exp_z / np.sum(exp_z, axis=axis, keepdims=True)

In [3]:
def causal_mask(n):
    mask = np.zeros((n, n), dtype=float)
    mask[np.triu_indices(n, k=1)] = -np.inf
    return mask

In [4]:
def attention(Q, K, V, mask=None):
    d = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d)

    if mask is not None:
        scores = scores + mask

    weights = softmax(scores, axis=-1)
    out = weights @ V
    return out, weights

In [5]:
out, w = attention(Q, K, V, causal_mask(4))

print(out.shape, w.shape)
print(np.round(w.sum(axis=1), 6))
print(np.round(w[0], 3))
print(np.allclose(np.triu(w, 1), 0))

(4, 2) (4, 4)
[1. 1. 1. 1.]
[1. 0. 0. 0.]
True
